In [ ]:
from pathlib import Path
import json
import random

In [ ]:
def get_project_root() -> Path:
    current = Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / "data_preprocess").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Unable to locate project root.")

PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "data_mixing"

base_token = 200_000_000
experiment_name = "exp2"
validation_size = 1_000

SPLIT_RATIOS = {
    "0.125": 0.125,
    "0.25": 0.25,
    "0.375": 0.375,
    "0.5": 0.5,
    "0.625": 0.625,
    "0.75": 0.75,
}

token_limits = {label: int(base_token * ratio) for label, ratio in SPLIT_RATIOS.items()}

DOMAIN_DATASETS = {
    "code": DATA_DIR / "opencoder-sft_len.json",
    "instr": DATA_DIR / "Infinity-Instruct_0625_len.json",
    "math": DATA_DIR / "openmathinstruct2_1M_len.json",
}

def sample_tokens(data, token_limit):
    selected = []
    total_tokens = 0
    for item in data:
        if total_tokens + item["len"] <= token_limit:
            total_tokens += item["len"]
            selected.append({k: v for k, v in item.items() if k != "len"})
        else:
            remaining_tokens = token_limit - total_tokens
            truncated_item = {k: v for k, v in item.items() if k != "len"}
            truncated_item["output"] = truncated_item.get("output", "")[:remaining_tokens]
            selected.append(truncated_item)
            break
    return selected

Total tokens for the sampled 323000 items: 174044404


In [ ]:
experiment_dir = OUTPUT_DIR / experiment_name
experiment_dir.mkdir(parents=True, exist_ok=True)

combined_validation = []

for domain, dataset_path in DOMAIN_DATASETS.items():
    with dataset_path.open("r") as f:
        structured_data = json.load(f)

    validation_set = random.sample(structured_data, validation_size)
    validation_ids = {json.dumps(item, sort_keys=True) for item in validation_set}
    filtered_validation = [{k: v for k, v in item.items() if k != "len"} for item in validation_set]

    val_filename = f"{base_token}_{domain}_val.json"
    val_path = experiment_dir / val_filename
    with val_path.open("w") as f:
        json.dump(filtered_validation, f, indent=2)

    combined_validation.extend(filtered_validation)

    filtered_data = [item for item in structured_data if json.dumps(item, sort_keys=True) not in validation_ids]
    random.shuffle(filtered_data)

    for label, limit in token_limits.items():
        sampled_items = sample_tokens(filtered_data, limit)
        subset_path = experiment_dir / f"{base_token}_{domain}_{label}.json"
        with subset_path.open("w") as f:
            json.dump(sampled_items, f, indent=2)

        print(f"Saved {subset_path.name} with {len(sampled_items)} items")

combined_name = f"{base_token}_{experiment_name}_val"
combined_path = experiment_dir / f"{combined_name}.json"
with combined_path.open("w") as f:
    json.dump(combined_validation, f, indent=2)

print(f"Saved {combined_path.name} with {len(combined_validation)} items")

In [ ]:
dataset_info_path = DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with dataset_info_path.open("r") as f:
        try:
            dataset_info = json.load(f)
        except json.JSONDecodeError:
            dataset_info = {}
else:
    dataset_info = {}

for domain in DOMAIN_DATASETS:
    val_name = f"{base_token}_{domain}_val"
    val_relative = Path("data_mixing") / experiment_name / f"{val_name}.json"
    dataset_info[val_name] = {"file_name": str(val_relative)}

    for label in token_limits:
        dataset_name = f"{base_token}_{domain}_{label}"
        relative_output = Path("data_mixing") / experiment_name / f"{dataset_name}.json"
        dataset_info[dataset_name] = {"file_name": str(relative_output)}

combined_name = f"{base_token}_{experiment_name}_val"
combined_relative = Path("data_mixing") / experiment_name / f"{combined_name}.json"
dataset_info[combined_name] = {"file_name": str(combined_relative)}

with dataset_info_path.open("w") as f:
    json.dump(dataset_info, f, indent=2)

Validation set created with 1000 items.


Validation set created with 1000 items.


Validation set created with 1000 items.


Sampled 54088 items for 0.125 with token limit 25000000.
Sampled 108121 items for 0.25 with token limit 50000000.
Sampled 162590 items for 0.375 with token limit 75000000.
Sampled 216860 items for 0.5 with token limit 100000000.
Sampled 271230 items for 0.625 with token limit 125000000.
Sampled 325621 items for 0.75 with token limit 150000000.


Sampled 46572 items for 0.125 with token limit 25000000.
Sampled 92830 items for 0.25 with token limit 50000000.
Sampled 139301 items for 0.375 with token limit 75000000.
Sampled 185512 items for 0.5 with token limit 100000000.
Sampled 232035 items for 0.625 with token limit 125000000.
Sampled 278624 items for 0.75 with token limit 150000000.


Sampled 56643 items for 0.125 with token limit 25000000.
Sampled 113084 items for 0.25 with token limit 50000000.
Sampled 169488 items for 0.375 with token limit 75000000.
Sampled 225612 items for 0.5 with token limit 100000000.
Sampled 282313 items for 0.625 with token limit 125000000.
Sampled 338823 items for 0.75 with token limit 150000000.


3000
